In [ ]:
    from konlpy.tag import Okt
    from collections import Counter
    import re
    from langchain import PromptTemplate, LLMChain
    from langchain.chat_models import ChatOpenAI
    from dotenv import load_dotenv
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = "all"
    okt = Okt()

    # 불용어 정의
    stopwords = set(["그", "저", "것", "이", "저는", "제가", "근데", "좀", "그냥", "정말", "되게", "음", "뭐"])

    # 텍스트 정제
    def clean_text(text):
        text = re.sub(r"[^가-힣\s]", "", text)
        return text.strip()

    # 자주 쓰는 단어 추출
    def extract_keywords(texts, min_len=2, top_k=20):
        counter = Counter()
        for text in texts:
            text = clean_text(text)
            nouns = okt.nouns(text)
            nouns = [n for n in nouns if len(n) >= min_len and n not in stopwords]
            counter.update(nouns)
        return counter.most_common(top_k)

    # 문장 끝 표현 추출
    def extract_sentence_endings(texts):
        endings = []
        for text in texts:
            sentences = re.split(r'[.!?]', text)
            for sentence in sentences:
                sentence = sentence.strip()
                if sentence:
                    morphs = okt.morphs(sentence)
                    if morphs:
                        endings.append(morphs[-1])
        return Counter(endings).most_common(10)

    # 종합 분석 함수
    def analyze_user_style(texts):
        result = {}
        result["keywords"] = extract_keywords(texts)
        result["endings"] = extract_sentence_endings(texts)
        return result

    # 분석 결과 출력
    def print_analysis(result):
        print("🧠 [말버릇 분석 결과]\n")

        # 자주 사용하는 단어
        print("\n 자주 사용하는 단어 (상위 5개):")
        if result["keywords"]:
            for word, freq in result["keywords"][:5]:
                print(f"  - {word}: {freq}회")
        else:
            print("  ❌ 없음")

        # 문장 끝 표현
        print("\n 문장 끝 표현 (상위 5개):")
        if result["endings"]:
            for ending, count in result["endings"][:5]:
                print(f"  - '~{ending}': {count}회")
        else:
            print("  ❌ 없음")

        print("\n✅ 분석 완료\n")

    # === 메인 로직 ===
    if __name__ == "__main__":
    # 다른 모듈/함수에서 받아온 음성 텍스트 변환 결과
    #음성 텍스트로 변환한걸 stt_result_text로 불러오면 될 것 같습니다
    stt_result_text = "안녕하세요 저는 오늘 기분이 아주 좋습니다. 그리고 뭐뭐 했고요. 정말 즐거운 하루였네요."

    if not stt_result_text.strip(): # 받아온 텍스트가 비어있는지 확인
        print("⛔ 분석할 텍스트가 없습니다.")
    else:
        user_texts = [stt_result_text] # STT 결과를 리스트에 담아 전달
        analysis = analyze_user_style(user_texts)

        # 분석 결과 출력
        print_analysis(analysis)

        # LangChain LLM 피드백 생성
        load_dotenv()
        llm = ChatOpenAI(model="gpt-4o", temperature=0.7, max_tokens=1000)

        template = """
        당신은 한국어 말투 및 화법 전문가입니다.
        아래는 사용자의 말버릇 분석 결과입니다:
        - 자주 사용하는 단어: {keywords}
        - 문장 끝 표현: {endings}

        다음과 같은 형식으로 자세하고 구체적인 피드백을 작성해 주세요:

        1) **특징 분석**: 사용자가 자주 쓰는 단어와 문장 끝 표현이 의미하는 바를 구체적으로 해석하고, 말투의 느낌(예: 친근함, 격식, 강약 등)을 분석해 주세요.

        2) **개선 방안**: 표현을 다양하게 하는 방법, 말투를 자연스럽게 만드는 팁, 어색한 표현을 피하는 법 등 구체적인 조언을 여러 항목으로 나누어 설명해 주세요.

        3) **예시 문장**: 실제 사용자가 쓴 문장과 개선된 문장을 3~5쌍 정도 예시로 들어, 차이점과 개선 포인트를 명확하게 보여 주세요.

        ⚠️ 응답은 위 형식을 엄격히 준수해 주세요. 분석 결과나 다른 설명 없이, 오직 피드백 내용만 출력해 주세요.
        """
        prompt = PromptTemplate(input_variables=["keywords","endings"], template=template)
        chain = LLMChain(llm=llm, prompt=prompt)

        keywords_str = ", ".join([f"{word}({count}회)" for word, count in analysis["keywords"]])
        endings_str = ", ".join([f"{ending}({count}회)" for ending, count in analysis["endings"]])

        result = chain.invoke({
            "keywords": keywords_str,
            "endings": endings_str
        })

        if isinstance(result, dict) and "text" in result:
            print(result["text"])
        elif hasattr(result, "content"):
            print(result.content)
        else:
            print(result)

⛔ 입력된 문장이 없습니다.
